# Training the 915M model on Kaggle

Sized for **T4 x2 (15 GB each), 30 GB RAM, 12-hour sessions.**

Sidebar: **Accelerator → GPU T4 x2**, **Internet → On**.

### The memory arithmetic

The model is 914,729,472 parameters. Whether it fits a 15 GB card is decided
entirely by the optimizer:

```
AdamW                weights 3.7 + grads 3.7 + m 3.7 + v 3.7 = 14.6 GB   no
SGD, momentum 0.9    weights 3.7 + grads 3.7 + buffer 3.7    = 11.0 GB   tight
SGD, momentum 0      weights 3.7 + grads 3.7                 =  7.3 GB   yes
```

**`--momentum 0` is load-bearing.** Momentum allocates one extra full copy of
the parameters, and it used to be hardcoded at 0.9 with no way to change it —
which is why an earlier attempt died inside `optimizer.step()` after the
forward and backward had already succeeded.

Only one of the two T4s is used: this project has no DDP or FSDP.


## 1. Hardware


In [ ]:
import os, subprocess, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print(subprocess.run(['nvidia-smi','--query-gpu=index,name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 2. Code


In [ ]:
%cd /kaggle/working
!rm -rf ai_from_zero
!git clone -q --branch main-5h8bvh --single-branch \
    https://github.com/m9cherif/ai_from_zero.git
%cd /kaggle/working/ai_from_zero
!pip install -q -r requirements.txt
from myai.core.device import setup, describe
print(describe(setup('auto')))


## 3. Choose your training data

Two options. **A** is the built-in corpus (~100 public-domain books plus
WikiText-2, 25M tokens) — enough for the 8.3M model, but the 915M loops
through it more than twice in one session.

**B** searches the Hub and downloads to a size budget you set. For a 12-hour
T4 session at ~1,500 tokens/s you can reach roughly 55M tokens, so **2 GB
(~540M tokens) means the model never sees the same text twice.**

| Budget | Tokens | Notes |
|---|---|---|
| 500 MB | ~135M | comfortable, still repeats a little |
| 2 GB | ~540M | no repetition in one session |
| 10 GB | ~2.7B | enough for several chained sessions |


In [ ]:
# --- A: the built-in corpus -------------------------------------------
# !python scripts/fetch_corpus.py

# --- B: pick from the Hub by budget (recommended for 915M) -------------
!python scripts/list_datasets.py --limit 60 --scan 1200 --out catalogue.json --show 12
!python scripts/fetch_corpus.py --hf-from catalogue.json \
    --budget-mb 2000 --max-per-dataset 4 --skip-gutenberg


## 4. Tokenizer

Must report `lossless decode: True`. On a multi-gigabyte corpus this fits the
vocabulary on a sample rather than the whole thing, which is standard.


In [ ]:
!python scripts/build_tokenizer.py --type bpe --vocab-size 4096 \
    --data 'data/train/*.txt'


## 5. Measure speed before setting a step budget

`--steps` sets the cosine learning-rate schedule, so it has to be roughly
right from the start. Run 60 steps, read the reported `tokens/s`, then use
the next cell.


In [ ]:
!python scripts/train.py --preset xl \
    --data data/train --cache-dir output/tokens \
    --steps 60 --batch-size 8 --lr 0.05 \
    --optimizer sgd --momentum 0 --grad-checkpoint --no-save-optimizer \
    --dtype float16 --mixed-precision --flash --device auto \
    --log-every 10 --save-every 100000 --output /kaggle/working/probe


In [ ]:
MEASURED_TOKENS_PER_SEC = 1500    # <- from the probe above
BATCH, SEQ, HOURS = 8, 256, 10.5  # 10.5 leaves time for eval and upload

steps = int(MEASURED_TOKENS_PER_SEC * HOURS * 3600 / (BATCH * SEQ))
tokens = steps * BATCH * SEQ
print(f'steps            : {steps:,}')
print(f'tokens seen      : {tokens/1e6:,.0f}M')
print(f'tokens/parameter : {tokens/914_729_472:.3f}   (Chinchilla ~20)')


## 6. Train

Checkpoints go to `/kaggle/working/xl`, which becomes the notebook output.
`--save-every 2000` matters: the session is killed at 12 hours regardless of
progress, and anything unwritten is lost.

To continue a previous run, uncomment `--resume`. It accepts a local path or
an `hf://` address, so a chained session can resume straight from the Hub.


In [ ]:
STEPS = 25000    # <- from the cell above

!python scripts/train.py --preset xl \
    --data data/train --val-data data/val --cache-dir output/tokens \
    --steps {STEPS} --batch-size 8 --lr 0.05 --dropout 0.0 \
    --optimizer sgd --momentum 0 --grad-checkpoint --no-save-optimizer \
    --dtype float16 --mixed-precision --flash --device auto \
    --log-every 100 --save-every 2000 --output /kaggle/working/xl
#   --resume hf://YOURNAME/myai-xl/checkpoint_latest.pt


**Out of memory?** Drop `--batch-size` to 4, then 2. Gradient checkpointing is
already on and `--momentum 0` already saves 3.7 GB, so batch size is the
remaining lever. If VRAM sits well under 14 GB, raise it to 16 instead.


## 7. Score and sample


In [ ]:
!python scripts/evaluate.py --data data/val --max-batches 200 --device auto \
    --checkpoint /kaggle/working/xl/checkpoint_latest.pt


In [ ]:
!python scripts/chat.py --max-tokens 60 --device auto \
    --checkpoint /kaggle/working/xl/checkpoint_latest.pt --prompt \
    'The old man walked into the' 'She said that' 'It was the best of'


## 8. Publish the checkpoint — do not skip this

Kaggle deletes `/kaggle/working` when the session ends. A 3.5 GB checkpoint
also cannot live in the GitHub repository (100 MB per file), so **the code
lives on GitHub and the weights live on the Hub.**

Add your token first: **Add-ons → Secrets → `HF_TOKEN`** (a write token from
huggingface.co → Settings → Access Tokens). Never paste it into a cell — a
notebook you share carries its output with it.


In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

!pip install -q huggingface_hub
!python scripts/push_model.py --repo YOURNAME/myai-xl \
    --checkpoint /kaggle/working/xl/checkpoint_latest.pt \
    --tokenizer output/tokenizer.json


### Now it works everywhere

Once published, every entry point takes the address instead of a path and
caches on first use — on your laptop, in the next Kaggle session, anywhere:

```bash
python scripts/chat.py     --checkpoint hf://YOURNAME/myai-xl/checkpoint_latest.pt
python scripts/serve.py    --checkpoint hf://YOURNAME/myai-xl/checkpoint_latest.pt
python scripts/evaluate.py --checkpoint hf://YOURNAME/myai-xl/checkpoint_latest.pt --data data/val
python scripts/train.py    --resume     hf://YOURNAME/myai-xl/checkpoint_latest.pt --steps 50000
```

That last line is how you chain sessions: each run resumes the published
model, trains further, and publishes again. The step counter carries over, so
raise `--steps` to the new cumulative total each time.

A backup inside Kaggle also works — **Save Version**, then **+ Add Input →
Notebook Output** next session — but the Hub address is what makes the model
usable off Kaggle.


## What to expect

Ten hours on a T4 is roughly 55M tokens: about **0.06 tokens per parameter**
against the ~20 that trains this size properly. It is still 35 times more
training than the model has had, and the loss should finally move — it has sat
at 6.93 across eight measurements spanning a sevenfold range of training,
because a few hundred steps teach it nothing beyond which words are common.

**Watch the first thousand steps: below 6.9 means the GPU is doing something
the CPU runs never could.**

It will not overtake the small model on this data. An 8.3M model on the same
corpus reached held-out perplexity **47.8** and writes real sentences; the
915M reached **1,022.8**. That gap closes only with far more data, which is
what the 2 GB budget in section 3 is for.

---

[m9cherif/ai_from_zero](https://github.com/m9cherif/ai_from_zero) · branch
`main-5h8bvh` · 186 tests ·
[`docs/SCALING.md`](https://github.com/m9cherif/ai_from_zero/blob/main-5h8bvh/docs/SCALING.md)
